# Data Fusion 2025 — Задача 2 "4cast"
## Прогнозирование еженедельных переводов юридических лиц

**Задача:** предсказать суммарные переводы 51 963 клиентов банка за 12 недель (118–129),  
используя историю за 118 недель (0–117).

**Метрика:** средний по клиентам RMSLE:
$$\overline{RMSLE} = \frac{1}{N}\sum_{i=1}^N \sqrt{\frac{1}{T}\sum_{t=1}^T (\log(1+y_{it}) - \log(1+\hat{y}_{it}))^2}$$

**Public leaderboard:** недели 118–121 (4 из 12 прогнозных).

---


## Скачивание данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import warnings
import json
from pathlib import Path
import gc, time

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4
sns.set_palette("tab10")

DATA_DIR = Path("data")
FIG_DIR  = Path("figures1")
FIG_DIR.mkdir(exist_ok=True)

# Splits (из calendar_extended.csv)
TRAIN_WEEKS    = list(range(0, 106))     # 0–105  → train
VAL_PUB_WEEKS  = list(range(106, 110))  # 106–109 → validation_public
VAL_PRIV_WEEKS = list(range(110, 118))  # 110–117 → validation_private
PUBLIC_WEEKS   = list(range(118, 122))  # 118–121 → public leaderboard
PRIVATE_WEEKS  = list(range(122, 130))  # 122–129 → private leaderboard
FORECAST_WEEKS = list(range(118, 130))  # все 12 недель прогноза

print("Config loaded.")


Config loaded.


In [2]:
print("Loading data...")
ts_train = pd.read_parquet(DATA_DIR / "target_series_extended.parquet")       # weeks 0–117
calendar = pd.read_csv(DATA_DIR / "calendar_extended.csv", parse_dates=["date"])
sample   = pd.read_csv(DATA_DIR / "sample_submit_extended.csv")

ts_train["week"] = ts_train["week"].astype(int)

TX_FILES = sorted(DATA_DIR.glob("transactions_[1-5].parquet"))
print(f"Found {len(TX_FILES)} transaction files: {[f.name for f in TX_FILES]}")

t0 = time.time()
trns_chunks = []
for f in TX_FILES:
    print(f"Reading {f.name} ...")
    chunk = pd.read_parquet(f)
    print(f"  -> {len(chunk):,} rows")
    trns_chunks.append(chunk)

trns_raw = pd.concat(trns_chunks, ignore_index=True)
del trns_chunks
gc.collect()

Loading data...
Found 5 transaction files: ['transactions_1.parquet', 'transactions_2.parquet', 'transactions_3.parquet', 'transactions_4.parquet', 'transactions_5.parquet']
Reading transactions_1.parquet ...
  -> 52,089,892 rows
Reading transactions_2.parquet ...
  -> 55,045,806 rows
Reading transactions_3.parquet ...
  -> 59,510,350 rows
Reading transactions_4.parquet ...
  -> 61,492,247 rows
Reading transactions_5.parquet ...
  -> 30,036,951 rows


0

## Подготовка данных 

Разбиваем датасет с транзакциями на 117 файлов, по неделе на файл для оптимизации. 

In [3]:
trns = trns_raw.copy()

In [4]:
#trns = trns.iloc[:50000]

In [5]:
trns["date"] = pd.to_datetime(trns["date"], utc=True).dt.tz_localize(None).dt.normalize()

all_inn = ts_train["inn_id"].unique()

cal_117 = calendar.loc[calendar["week"] <= 117].copy()
cal_117["date"] = pd.to_datetime(cal_117["date"]).dt.tz_localize(None).dt.normalize()
all_dates = np.sort(cal_117["date"].drop_duplicates().values)

mi = pd.MultiIndex.from_product([all_inn, all_dates], names=["doc_payer_inn", "date"])
new_trns = mi.to_frame(index=False)

new_trns["doc_payee_inn"] = ""
new_trns["trns_count"] = 1.0
new_trns["trns_amount"] = 0
new_trns["doc_payer_bank_name_encoded"] = 51
new_trns["doc_payee_bank_name_encoded"] = 0
new_trns["doc_payer_bank_name_flag"] = 1
new_trns["doc_payee_bank_name_flag"] = 0
new_trns["trns_class_encoded"] = 0

trns = pd.concat([trns, new_trns], ignore_index=True)

trns = pd.merge(trns, calendar, on='date', how='left')

allowed = ts_train["inn_id"].unique()
trns = trns[trns["doc_payer_inn"].isin(allowed)]

In [6]:
trns = trns.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "target",
             })

In [10]:
trns.head()

,week,inn_id,target,part
0,0,inn1000051,4.221593e+05,train
1,0,inn1000367,3.994532e+05,train
2,0,inn1000484,9.672847e+05,train
3,0,inn1000624,1.684209e+05,train
4,0,inn1000884,5.724560e+06,train


In [12]:
expected_weeks = sorted(trns["week"].astype(int).unique().tolist())

expected_len = trns["inn_id"].nunique()

dup_mask = trns.duplicated(subset=["inn_id", "week"], keep=False)
if dup_mask.any():
    dup_count = int(dup_mask.sum())
    raise ValueError(f"Найдены дубли по (inn_id, week): {dup_count} строк")

min_week, max_week = expected_weeks[0], expected_weeks[-1]
full_range = list(range(min_week, max_week + 1))
if expected_weeks != full_range:
    missing_weeks = sorted(set(full_range) - set(expected_weeks))
    raise ValueError(f"Неполный диапазон недель. Отсутствуют: {missing_weeks}")

out_dir = Path("transactions_chunks")
out_dir.mkdir(parents=True, exist_ok=True)

trns_sorted = trns.sort_values(["week", "inn_id"]).reset_index(drop=True)
errors = []
written_files = 0

for week, chunk in trns_sorted.groupby("week", sort=True):
    chunk_len = len(chunk)
    if chunk_len != expected_len:
        errors.append(f"week={int(week)}: len={chunk_len}, expected={expected_len}")
        continue

    out_path = out_dir / f"week_{int(week):03d}.parquet"
    chunk.to_parquet(out_path, index=False)
    written_files += 1

if errors:
    preview = "\n".join(errors[:10])
    raise ValueError(f"Обнаружены недели с неверной длиной чанка:\n{preview}")

print(f"Done: записано {written_files} файлов в {out_dir.resolve()}")

Done: записано 118 файлов в /Users/aeseverenkova/Desktop/bank_forecasting/transactions_chunks


## Построение признаков

In [29]:
window = 70
start = 109 
validation_weeks = 14


In [30]:
features = pd.DataFrame()

for target_i in range(validation_weeks):
    start -= target_i
    weeks_in_window = range(start - window + 1, start) 
    df_test = trns[trns['week'].isin(weeks_in_window)]
    df_test = df_test.sort_values(["inn_id", "week"]).copy()

    max_lag = 12
    for lag in range(1, max_lag + 1):
        df_test[f"lag_{lag}"] =  df_test.groupby("inn_id")["target"].shift(lag)
        df_test["roll_mean"] = df_test.groupby("inn_id")["target"].shift(lag).transform(lambda s: s.rolling(window).mean())
        df_test["roll_min"] = df_test.groupby("inn_id")["target"].shift(lag).transform(lambda s: s.rolling(window).min())
        df_test["roll_max"] = df_test.groupby("inn_id")["target"].shift(lag).transform(lambda s: s.rolling(window).max())


    features = pd.concat([features, df_test])

features.tail(20)

,week,inn_id,target,part,lag_1,roll_mean,roll_min,roll_max,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,lag_11,lag_12
801457,16,inn999759,1.494178e+07,train,1.174501e+07,NaN,NaN,NaN,6.077322e+06,9.661978e+06,1.406870e+07,1.140155e+07,9.735985e+06,1.633000e+07,1.698672e+07,1.675813e+07,1.449805e+07,9.798962e+06,6.529082e+06
849971,17,inn999759,8.020993e+06,train,1.494178e+07,NaN,NaN,NaN,1.174501e+07,6.077322e+06,9.661978e+06,1.406870e+07,1.140155e+07,9.735985e+06,1.633000e+07,1.698672e+07,1.675813e+07,1.449805e+07,9.798962e+06
34977,0,inn999886,1.087004e+07,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82302,1,inn999886,5.854778e+06,train,1.087004e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130103,2,inn999886,3.097832e+06,train,5.854778e+06,NaN,NaN,NaN,1.087004e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
177053,3,inn999886,8.000828e+06,train,3.097832e+06,NaN,NaN,NaN,5.854778e+06,1.087004e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223823,4,inn999886,5.791335e+06,train,8.000828e+06,NaN,NaN,NaN,3.097832e+06,5.854778e+06,1.087004e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271320,5,inn999886,3.481578e+06,train,5.791335e+06,NaN,NaN,NaN,8.000828e+06,3.097832e+06,5.854778e+06,1.087004e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN
320431,6,inn999886,4.333780e+06,train,3.481578e+06,NaN,NaN,NaN,5.791335e+06,8.000828e+06,3.097832e+06,5.854778e+06,1.087004e+07,NaN,NaN,NaN,NaN,NaN,NaN
367580,7,inn999886,1.073443e+07,train,4.333780e+06,NaN,NaN,NaN,3.481578e+06,5.791335e+06,8.000828e+06,3.097832e+06,5.854778e+06,1.087004e+07,NaN,NaN,NaN,NaN,NaN


In [31]:
features.fillna(0, inplace=True)

numeric = ['target', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'roll_mean', 'roll_min', 'roll_max']
for feat in numeric:
    features[feat] = np.log(features[feat])

features.head(20)

,week,inn_id,target,part,lag_1,roll_mean,roll_min,roll_max,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,lag_11,lag_12
1918690,40,inn1000051,13.934799,train,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
1979499,41,inn1000051,11.890469,train,13.934799,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
5868857,42,inn1000051,-inf,train,11.890469,-inf,-inf,-inf,13.934799,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
2068006,43,inn1000051,10.988540,train,-inf,-inf,-inf,-inf,11.890469,13.934799,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
2118152,44,inn1000051,13.776553,train,10.988540,-inf,-inf,-inf,-inf,11.890469,13.934799,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
2218375,45,inn1000051,10.290271,train,13.776553,-inf,-inf,-inf,10.988540,-inf,11.890469,13.934799,-inf,-inf,-inf,-inf,-inf,-inf,-inf
2268077,46,inn1000051,13.371396,train,10.290271,-inf,-inf,-inf,13.776553,10.988540,-inf,11.890469,13.934799,-inf,-inf,-inf,-inf,-inf,-inf
2268973,47,inn1000051,13.048857,train,13.371396,-inf,-inf,-inf,10.290271,13.776553,10.988540,-inf,11.890469,13.934799,-inf,-inf,-inf,-inf,-inf
2319381,48,inn1000051,11.090250,train,13.048857,-inf,-inf,-inf,13.371396,10.290271,13.776553,10.988540,-inf,11.890469,13.934799,-inf,-inf,-inf,-inf
2369823,49,inn1000051,12.089892,train,11.090250,-inf,-inf,-inf,13.048857,13.371396,10.290271,13.776553,10.988540,-inf,11.890469,13.934799,-inf,-inf,-inf


In [ ]:
# тест
df_test = trns[trns['week'].isin([109, 110, 111])]
df_test = df_test.sort_values(["inn_id", "week"]).copy()

max_lag = 12
for lag in range(1, max_lag + 1):
    df_test[f"lag_{lag}"] =  df_test.groupby("inn_id")["target"].shift(lag)

df_test.head(20)

,week,inn_id,target,part,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,lag_9,lag_10,lag_11,lag_12
5460859,109,inn1000051,2.508962e+06,validation_public,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5505065,110,inn1000051,5.958537e+06,validation_private,2.508962e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5565399,111,inn1000051,3.988973e+04,validation_private,5.958537e+06,2.508962e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5413749,109,inn1000246,8.807682e+06,validation_public,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5464410,110,inn1000246,8.224529e+06,validation_private,8.807682e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5515140,111,inn1000246,5.668363e+06,validation_private,8.224529e+06,8.807682e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5413750,109,inn1000281,1.863329e+05,validation_public,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5464411,110,inn1000281,7.764511e+05,validation_private,1.863329e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5515141,111,inn1000281,1.620226e+05,validation_private,7.764511e+05,1.863329e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5413751,109,inn1000367,9.940794e+05,validation_public,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Обучение

In [11]:
!pip3 install catboost
!pip3 install scikit-learn

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [12]:
import numpy as np
from sklearn import metrics
import pandas as pd
from catboost import CatBoostRegressor

In [13]:
FEATURE_COLS = [
    "lag_1", "lag_7", "lag_14", "lag_28", "roll_mean_7", "dow", "month", "weekend"
]


def fit_and_predict(train_frame: pd.DataFrame, valid_frame: pd.DataFrame):
    X_train = train_frame[FEATURE_COLS]
    y_train = train_frame["target_daily"]
    X_valid = valid_frame[FEATURE_COLS]
    y_valid = valid_frame["target_daily"]

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=1000,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=False,
    )
    model.fit(X_train, np.log1p(y_train))

    pred_log = model.predict(X_valid)
    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)
    return y_valid, pred


def group_rmsle(frame: pd.DataFrame) -> float:
    by_client = frame.groupby("inn_id", sort=False)
    client_scores = by_client.apply(
        lambda g: np.sqrt(metrics.mean_squared_log_error(g["y_true"], g["y_pred"]))
    )
    return float(client_scores.mean())

# без leakage
fixed_df = df_daily.sort_values(["inn_id", "date"]).copy()
grouped = fixed_df.groupby("inn_id", sort=False)

fixed_df["lag_1"] = grouped["target_daily"].shift(1)
fixed_df["lag_7"] = grouped["target_daily"].shift(7)
fixed_df["lag_14"] = grouped["target_daily"].shift(14)
fixed_df["lag_28"] = grouped["target_daily"].shift(28)
fixed_df["roll_mean_7"] = grouped["target_daily"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).mean()
)
fixed_df["dow"] = fixed_df["date"].dt.dayofweek
fixed_df["month"] = fixed_df["date"].dt.month
fixed_df["weekend"] = fixed_df["dow"].isin([5, 6])

fixed_data = fixed_df.dropna(subset=FEATURE_COLS + ["target_daily"]).copy()
train_mask = fixed_data["week"] <= 109
valid_mask = fixed_data["week"].between(110, 117)

train_df = fixed_data.loc[train_mask].copy()
valid_df = fixed_data.loc[valid_mask].copy()

y_valid, pred = fit_and_predict(train_df, valid_df)

global_rmsle_fixed = float(np.sqrt(metrics.mean_squared_log_error(y_valid, pred)))

pred_frame = valid_df[["inn_id", "date"]].copy()
pred_frame["y_true"] = y_valid.values
pred_frame["y_pred"] = pred
group_rmsle_fixed = group_rmsle(pred_frame)

print(f"A: {global_rmsle_fixed}")
print(f"RMSLE: {group_rmsle_fixed}")

A: 4.509756162583145
RMSLE: 4.407655346201267


In [18]:
pred

array([1.18132077e+02, 9.72280843e+01, 2.26658308e+03, ...,
       1.81658876e+06, 1.80509779e+02, 1.57568572e+01], shape=(2909928,))